# training.ipynb

Notebook ini dijalankan pada benchmark riil 2021-2023 dan harus dibaca bersama artefak provenance, comparison report, serta manual review summary.

Benchmark saat ini memakai slice data riil tahun 2021-2023; lihat `data/processed/source_manifest.json`, `models/benchmark_comparison.json`, dan `data/processed/manual_review_summary.csv` untuk konteks synthetic-vs-real dan dampak manual review.

Jika row-level reviewed labels tersedia nanti, impor dengan `scripts/import_reviewed_row_level.py` lalu rerun `scripts/run_diagnostics.py` agar artefak reviewed benchmark ikut terbarui.


In [3]:
from pathlib import Path
import json
import pandas as pd
import xgboost as xgb
from src.model import compute_sample_weights

train_X = pd.read_parquet("train_data/features.parquet")
train_y = pd.read_parquet("train_data/labels.parquet")["risk_label"]
test_X = pd.read_parquet("test_data/features.parquet")
test_y = pd.read_parquet("test_data/labels.parquet")["risk_label"]
print(train_X.shape, test_X.shape)


(372150, 34) (93034, 34)


In [2]:
params = json.loads(Path("models/best_params.json").read_text())
n_rounds = int(params.pop("n_rounds", 449))
weights = compute_sample_weights(train_y)
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    seed=42,
    n_estimators=n_rounds,
    n_jobs=-1,
    **params,
)
model.fit(train_X, train_y, sample_weight=weights)
model.save_model("models/xgb_model.ubj")
print("saved", Path("models/xgb_model.ubj").exists())


FileNotFoundError: [Errno 2] No such file or directory: 'models/best_params.json'

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
preds = model.predict(test_X)
print({
    "accuracy": round(float(accuracy_score(test_y, preds)), 4),
    "macro_f1": round(float(f1_score(test_y, preds, average="macro")), 4),
    "weighted_f1": round(float(f1_score(test_y, preds, average="weighted")), 4),
})
print(classification_report(test_y, preds))


In [ ]:
renamed = train_X.copy()
renamed.columns = [f"f{i}" for i in range(renamed.shape[1])]
onnx_model_src = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    seed=42,
    n_estimators=n_rounds,
    n_jobs=-1,
    **params,
)
onnx_model_src.fit(renamed, train_y, sample_weight=weights)
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
initial_type = [("float_input", FloatTensorType([None, renamed.shape[1]]))]
onnx_model = convert_xgboost(onnx_model_src, initial_types=initial_type)
with open("models/xgb_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
print("saved", Path("models/xgb_model.onnx").exists())
